# Purpose

This notebook implements the **evaluation stage** of the fetal vein segmentation project. It scores predicted masks against ground truth using the lecturer workflow: metrics without post-processing (`No`) and with post-processing (`Yes`).

The morphological operation applied in the `Yes` branch is selected via `POSTPROCESS_METHOD` (erosion, dilation, opening, closing). Compare operations by re-running the notebook with a different method and comparing the exported CSV files.

**Inputs:** predicted masks in `02_dataset/results_*` and ground-truth labels in `02_dataset/labels/`.
**Outputs:** an aggregated comparison table and `04_pipeline_results/tabela_avaliacao_experiencias_{POSTPROCESS_METHOD}.csv` for interpretation in the written report.


# Dependencies

All third-party and standard-library imports for this notebook are declared here.
Downstream cells must not repeat these imports.

## Import groups

Standard library, NumPy, Pandas, Matplotlib, SciPy morphology, and display helpers.


In [ ]:
%matplotlib inline

# --- Standard library ---
import re
from pathlib import Path

# --- Numerical computing ---
import numpy as np
import pandas as pd

# --- Image I/O and visualisation ---
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from IPython.display import display

# --- Morphological post-processing ---
from scipy.ndimage import (
    binary_dilation,
    binary_erosion,
    binary_opening,
    label,
)


## Project paths and pairing utilities

**Purpose:** Resolve repository root; map predictions to labels by patient identifier.
**Provenance:** pairing and path helpers (consolidated into this notebook).


In [ ]:
def find_project_root(start_path: Path) -> Path:
    """Walk up the directory tree until 02_dataset/ is found."""
    for candidate_path in [start_path, *start_path.parents]:
        if (candidate_path / "02_dataset").is_dir():
            return candidate_path
    raise FileNotFoundError(
        f"Directory 02_dataset/ not found from {start_path}. "
        "Run the notebook from the fetal_vein_segmentation repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "02_dataset"
IMAGES_ORIGINAL_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"
SAVE_MODELS_DIR = DATA_DIR / "Save_Models"

PREPROCESSED_IMAGE_DIRS = [
    DATA_DIR / "images_pp_1",
    DATA_DIR / "images_pp_2",
    DATA_DIR / "images_pp_3",
    DATA_DIR / "images_pp_4",
    DATA_DIR / "images_pp_5",
]

RESULT_FOLDER_CONFIG = [
    ("Original", DATA_DIR / "results_original"),
    ("PP1", DATA_DIR / "results_pp_1"),
    ("PP2", DATA_DIR / "results_pp_2"),
    ("PP3", DATA_DIR / "results_pp_3"),
    ("PP4", DATA_DIR / "results_pp_4"),
    ("PP5", DATA_DIR / "results_pp_5"),
]

EXPECTED_MODEL_FILES = [
    "best_metric_model_original.pth",
    "best_metric_model_pp_1.pth",
    "best_metric_model_pp_2.pth",
    "best_metric_model_pp_3.pth",
    "best_metric_model_pp_4.pth",
    "best_metric_model_pp_5.pth",
]

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

PP_SUFFIX_PATTERN = re.compile(r"_PP_PL_\d+$", re.IGNORECASE)
PATIENT_ID_PATTERN = re.compile(r"^P\d+", re.IGNORECASE)



### Pairing and validation functions

Define how prediction filenames map to ground-truth labels and how repository folders are checked before metrics run.
These utilities mirror the lecturer pairing contract (patient identifier, not list index).



In [ ]:
def extract_base_identifier(file_name: str) -> str:
    """
    Original image identifier (without preprocessing suffix).
    Ex.: P080_IMG1_PP_PL_1.png → P080_IMG1; P080_IMG1.png → P080_IMG1.
    """
    stem = Path(file_name).stem
    original_identifier = PP_SUFFIX_PATTERN.sub("", stem)
    if not original_identifier or not PATIENT_ID_PATTERN.match(original_identifier):
        raise ValueError(
            f"Invalid original identifier in '{file_name}'. "
            f"Expected format P080_IMG1 or P080_IMG1_PP_PL_N."
        )
    return original_identifier



In [ ]:
def resolve_label_path(prediction_file_name: str, labels_dir: Path = LABELS_DIR) -> Path:
    """Maps image/prediction to ground truth in labels/ using the original identifier."""
    original_identifier = extract_base_identifier(prediction_file_name)
    candidate_path = labels_dir / f"{original_identifier}.png"
    if candidate_path.is_file():
        return candidate_path
    raise FileNotFoundError(
        f"Missing label for '{prediction_file_name}' (ID original {original_identifier}). "
        f"Expected: {candidate_path}"
    )



In [ ]:
def list_predictions(results_folder: Path) -> list:
    """Lists prediction files sorted by name (does not define pairing)."""
    if not results_folder.is_dir():
        return []
    return sorted(
        p
        for p in results_folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )



In [ ]:
def validate_project_paths() -> list:
    """Checks required folders; returns a list of warning strings."""
    warnings = []
    required_folders = [
        ("images (original)", IMAGES_ORIGINAL_DIR),
        ("labels", LABELS_DIR),
        ("Save_Models", SAVE_MODELS_DIR),
    ]
    for folder_name, folder_path in required_folders:
        if not folder_path.is_dir():
            warnings.append(f"Missing folder: {folder_name} → {folder_path}")

    for pipeline_index, folder_path in enumerate(PREPROCESSED_IMAGE_DIRS, start=1):
        if not folder_path.is_dir():
            warnings.append(f"Missing preprocessed dataset: images_pp_{pipeline_index}")
        elif len(list(folder_path.glob("*.png"))) == 0:
            warnings.append(f"images_pp_{pipeline_index} exists but contains no .png files")

    for experiment_key, results_folder in RESULT_FOLDER_CONFIG:
        if not results_folder.is_dir():
            warnings.append(f"Missing results folder: {experiment_key} → {results_folder.name}")
        elif len(list_predictions(results_folder)) == 0:
            warnings.append(f"{results_folder.name} exists but contains no predictions")

    for model_filename in EXPECTED_MODEL_FILES:
        model_path = SAVE_MODELS_DIR / model_filename
        if not model_path.is_file():
            warnings.append(f"Missing model: {model_filename}")

    return warnings



In [ ]:
def validate_pairing_folder(images_folder: Path, labels_dir: Path = LABELS_DIR) -> list:
    """Validates each image/prediction → label by ID (not by index)."""
    pairing_errors = []
    if not images_folder.is_dir():
        return [f"Folder does not exist: {images_folder}"]
    for image_path in sorted(images_folder.glob("*.png")):
        try:
            resolve_label_path(image_path.name, labels_dir)
        except (ValueError, FileNotFoundError) as exc:
            pairing_errors.append(f"{image_path.name}: {exc}")
    return pairing_errors



In [ ]:
def validate_all_pairings() -> dict:
    """Validates images, images_pp_*, and results folders."""
    pairing_report = {}
    pairing_report["images"] = validate_pairing_folder(IMAGES_ORIGINAL_DIR)
    for pipeline_index, folder_path in enumerate(PREPROCESSED_IMAGE_DIRS, start=1):
        pairing_report[f"images_pp_{pipeline_index}"] = validate_pairing_folder(folder_path)
    for experiment_key, results_folder in RESULT_FOLDER_CONFIG:
        pairing_errors = []
        for prediction_file in list_predictions(results_folder):
            try:
                resolve_label_path(prediction_file.name)
            except (ValueError, FileNotFoundError) as exc:
                pairing_errors.append(f"{prediction_file.name}: {exc}")
        pairing_report[f"results_{experiment_key}"] = pairing_errors
    return pairing_report



# 3. Post-Processing

**Purpose:** Refine predicted masks with mathematical morphology before metric computation in the `Yes` evaluation branch.

**Inputs:** Grayscale or binary prediction arrays from `02_segmentation.ipynb`.
**Outputs:** Binary `uint8` masks `{0, 1}`.
**Role:** Configurable operator via `POSTPROCESS_METHOD` and `apply_postprocessing()`; lecturer `pos_process()` retained as reference.

**Provenance:** post-processing helpers (consolidated into this notebook).


### `load_mask_png()`

**Description:** Load a PNG mask as a 2D array.
**Parameters:** `file_path` — path to mask file.
**Returns:** 2D NumPy array.
**Usage context:** Evaluation loop.



In [ ]:
def load_mask_png(file_path: Path) -> np.ndarray:
    """Loads a PNG mask as a 2D array."""
    loaded_image = mpimg.imread(file_path)
    if loaded_image.ndim == 3:
        loaded_image = loaded_image[:, :, 0]
    return loaded_image



### `binarize_mask()`

**Description:** Convert mask to binary {0, 1} uint8.
**Parameters:** `input_mask`, optional `threshold`.
**Returns:** Binary uint8 array.
**Usage context:** Used by morphological helpers, `apply_postprocessing()`, and `calculate_metrics`.



In [ ]:
def binarize_mask(input_mask: np.ndarray, threshold: float = 127.0) -> np.ndarray:
    """Converts mask to binary {0, 1} uint8."""
    if input_mask.dtype == np.bool_:
        return input_mask.astype(np.uint8)
    if input_mask.max() <= 1.0:
        return (input_mask > 0.5).astype(np.uint8)
    return (input_mask >= threshold).astype(np.uint8)



### `correct_prediction_orientation()`

**Description:** Orientation correction as in Pos_Metrics.ipynb.
**Parameters:** `prediction_array` — 2D mask.
**Returns:** Corrected 2D array.
**Usage context:** Applied before metrics when enabled in configuration.



In [ ]:
def correct_prediction_orientation(prediction_array: np.ndarray) -> np.ndarray:
    """
    Corrects prediction orientation (equivalent to Pos_Metrics.ipynb).
    predicted = np.flip(np.rot90(predicted, 1), 0)
    """
    return np.flip(np.rot90(prediction_array, 1), 0)



### Mathematical Morphology Background

Binary mathematical morphology analyses foreground regions by comparing a **binary mask** with a **structuring element** (SE). The SE acts as a local probe: each pixel is updated according to whether the SE, centred at that pixel, fits inside (erosion) or intersects (dilation) the foreground.

In segmentation post-processing, morphology can **smooth boundaries**, **remove small spurious regions**, or **fill small gaps** without retraining the model. All operations below assume a binarised prediction mask and a flat, square SE of size `MORPH_KERNEL_SIZE × MORPH_KERNEL_SIZE`.

### Morphological Operations

| Operation | Definition (same SE **B**) | Typical segmentation effect |
|-----------|----------------------------|-----------------------------|
| **Erosion** | \(A \ominus B\) | Shrinks foreground; removes thin protrusions |
| **Dilation** | \(A \oplus B\) | Expands foreground; closes small gaps |
| **Opening** | \(A \circ B = (A \ominus B) \oplus B\) | Removes small foreground artefacts |
| **Closing** | \(A \bullet B = (A \oplus B) \ominus B\) | Fills small holes inside foreground |

Opening and closing are implemented as **explicit compositions** of erosion and dilation. SciPy (`scipy.ndimage`) provides the discrete binary operators used as the computational backend.


### `pos_process()`

**Description:** Morphological opening followed by retention of the largest connected component.
**Parameters:** `predicted` mask; `opening_kernel_size` (default 3).
**Returns:** Post-processed binary uint8 mask.
**Usage context:** Lecturer reference (opening + largest component); not used in the No/Yes evaluation loop.
**Naming:** Lecturer identifier — do not rename.


In [ ]:
def pos_process(predicted: np.ndarray, opening_kernel_size: int = 3) -> np.ndarray:
    """
    Post-processing: morphological opening + largest connected component.

    Args:
        predicted: binary or grayscale mask.
        opening_kernel_size: square structuring element side length.

    Returns:
        Binary uint8 mask {0, 1}.
    """
    mask_binary = binarize_mask(predicted)
    structuring_element = np.ones((opening_kernel_size, opening_kernel_size), dtype=bool)
    opened_mask = binary_opening(mask_binary > 0, structure=structuring_element)

    labeled_mask, component_count = label(opened_mask)
    if component_count == 0:
        return np.zeros_like(mask_binary, dtype=np.uint8)

    largest_component_label = 1
    largest_component_area = 0
    for component_label in range(1, component_count + 1):
        component_area = np.sum(labeled_mask == component_label)
        if component_area > largest_component_area:
            largest_component_area = component_area
            largest_component_label = component_label

    predicted_pos_proc = (labeled_mask == largest_component_label).astype(np.uint8)
    return predicted_pos_proc


def build_structuring_element(kernel_size: int) -> np.ndarray:
    """
    Build a flat square structuring element for binary morphology.

    Parameters
    ----------
    kernel_size : int
        Side length of the square structuring element B (kernel_size × kernel_size).

    Returns
    -------
    np.ndarray
        Boolean structuring element; True denotes membership in B.

    Notes
    -----
    A flat SE treats all ones in B as a single translate of the origin.
    The same B is reused for erosion, dilation, opening, and closing.
    """
    return np.ones((kernel_size, kernel_size), dtype=bool)


def _foreground_boolean(mask: np.ndarray) -> np.ndarray:
    """Return boolean foreground array after binarisation."""
    return binarize_mask(mask) > 0


def apply_erosion(mask: np.ndarray, kernel_size: int = 3) -> np.ndarray:
    """
    Binary morphological erosion.

    Mathematical definition: A ⊖ B = { z | (B)_z ⊆ A }, where (B)_z is B
    translated so that its origin lies at pixel z.

    Parameters
    ----------
    mask : np.ndarray
        Prediction mask produced by the segmentation model (binary or grayscale).
    kernel_size : int
        Side length of the square structuring element.

    Returns
    -------
    np.ndarray
        Binary uint8 mask {0, 1} after erosion.

    Notes
    -----
    Erosion shrinks foreground regions by removing boundary pixels that
    cannot fully contain the structuring element. It suppresses thin
    structures and isolated noise. Implemented with scipy.ndimage.binary_erosion.
    """
    foreground = _foreground_boolean(mask)
    structuring_element = build_structuring_element(kernel_size)
    eroded = binary_erosion(foreground, structure=structuring_element)
    return eroded.astype(np.uint8)


def apply_dilation(mask: np.ndarray, kernel_size: int = 3) -> np.ndarray:
    """
    Binary morphological dilation.

    Mathematical definition: A ⊕ B = { z | (B)_z ∩ A ≠ ∅ }.

    Parameters
    ----------
    mask : np.ndarray
        Prediction mask produced by the segmentation model (binary or grayscale).
    kernel_size : int
        Side length of the square structuring element.

    Returns
    -------
    np.ndarray
        Binary uint8 mask {0, 1} after dilation.

    Notes
    -----
    Dilation expands foreground regions by adding pixels where the translated
    structuring element intersects the foreground. It can bridge small gaps
    between regions. Implemented with scipy.ndimage.binary_dilation.
    """
    foreground = _foreground_boolean(mask)
    structuring_element = build_structuring_element(kernel_size)
    dilated = binary_dilation(foreground, structure=structuring_element)
    return dilated.astype(np.uint8)


def apply_opening(mask: np.ndarray, kernel_size: int = 3) -> np.ndarray:
    """
    Binary morphological opening (composition of erosion and dilation).

    Mathematical definition: A ○ B = (A ⊖ B) ⊕ B using the same structuring
    element B for both steps.

    Parameters
    ----------
    mask : np.ndarray
        Prediction mask produced by the segmentation model (binary or grayscale).
    kernel_size : int
        Side length of the square structuring element.

    Returns
    -------
    np.ndarray
        Binary uint8 mask {0, 1} after opening.

    Notes
    -----
    Opening is defined as an erosion followed by a dilation with the same
    structuring element. This operation removes small foreground artefacts
    and thin protrusions while preserving larger structures. Implemented
    explicitly as apply_erosion followed by apply_dilation.
    """
    eroded = apply_erosion(mask, kernel_size=kernel_size)
    opened = apply_dilation(eroded, kernel_size=kernel_size)
    return opened


def apply_closing(mask: np.ndarray, kernel_size: int = 3) -> np.ndarray:
    """
    Binary morphological closing (composition of dilation and erosion).

    Mathematical definition: A • B = (A ⊕ B) ⊖ B using the same structuring
    element B for both steps.

    Parameters
    ----------
    mask : np.ndarray
        Prediction mask produced by the segmentation model (binary or grayscale).
    kernel_size : int
        Side length of the square structuring element.

    Returns
    -------
    np.ndarray
        Binary uint8 mask {0, 1} after closing.

    Notes
    -----
    Closing is defined as a dilation followed by an erosion with the same
    structuring element. This operation fills small holes and narrow gaps
    inside the foreground while preserving overall object size. Implemented
    explicitly as apply_dilation followed by apply_erosion.
    """
    dilated = apply_dilation(mask, kernel_size=kernel_size)
    closed = apply_erosion(dilated, kernel_size=kernel_size)
    return closed


def apply_postprocessing(mask: np.ndarray) -> np.ndarray:
    """
    Apply the morphological operator selected in POSTPROCESS_METHOD.

    Parameters
    ----------
    mask : np.ndarray
        Oriented prediction mask from the evaluation loop.

    Returns
    -------
    np.ndarray
        Binary mask after the configured morphological operation.

    Notes
    -----
    Reads POSTPROCESS_METHOD and MORPH_KERNEL_SIZE from the evaluation
    configuration cell. One operator is applied per notebook run; compare
    operators by re-running with a different POSTPROCESS_METHOD.
    """
    if POSTPROCESS_METHOD == "erosion":
        return apply_erosion(mask, kernel_size=MORPH_KERNEL_SIZE)
    if POSTPROCESS_METHOD == "dilation":
        return apply_dilation(mask, kernel_size=MORPH_KERNEL_SIZE)
    if POSTPROCESS_METHOD == "opening":
        return apply_opening(mask, kernel_size=MORPH_KERNEL_SIZE)
    if POSTPROCESS_METHOD == "closing":
        return apply_closing(mask, kernel_size=MORPH_KERNEL_SIZE)
    raise ValueError(
        f"Unknown POSTPROCESS_METHOD: {POSTPROCESS_METHOD!r}. "
        "Expected: erosion, dilation, opening, closing."
    )



# 4. Metrics

**Purpose:** Compute Dice, accuracy, precision, and recall on binarised masks.

**Provenance:** metrics workflow aligned with [Pos_Metrics.ipynb](01_academic/04_reference_materials/03_code_exemple/Pos_Metrics.ipynb).


### `calculate_metrics()`

**Description:** Binary segmentation metrics (lecturer naming).
**Parameters:** `predicted`, `gt` arrays.
**Returns:** Tuple `(dice, accuracy, precision, recall)`.
**Usage context:** Both evaluation branches (§5 and §6).
**Naming:** Lecturer identifier — do not rename.


In [ ]:
def calculate_metrics(predicted: np.ndarray, gt: np.ndarray):
    """
    Computes binary segmentation metrics.

    Returns:
        (dice, accuracy, precision, recall)
    """
    pred = binarize_mask(predicted)
    gt_bin = binarize_mask(gt)

    if pred.shape != gt_bin.shape:
        raise ValueError(
            f"Incompatible shapes: prediction {pred.shape} vs ground truth {gt_bin.shape}"
        )

    tp = int(np.sum((pred == 1) & (gt_bin == 1)))
    tn = int(np.sum((pred == 0) & (gt_bin == 0)))
    fp = int(np.sum((pred == 1) & (gt_bin == 0)))
    fn = int(np.sum((pred == 0) & (gt_bin == 1)))

    eps = 1e-8
    dice = (2.0 * tp) / (2.0 * tp + fp + fn + eps)
    ac = (tp + tn) / (tp + tn + fp + fn + eps)
    pr = tp / (tp + fp + eps)
    re = tp / (tp + fn + eps)

    return dice, ac, pr, re



### `compute_mean_metrics()`

**Description:** Mean of metric tuples across images.
**Parameters:** `metric_tuples` — list of (dice, ac, pr, re).
**Returns:** Dictionary of mean values.
**Usage context:** Aggregate per experiment in evaluation loop.



In [ ]:
def compute_mean_metrics(metric_tuples: list) -> dict:
    """Mean of (dice, accuracy, precision, recall) tuples."""
    if len(metric_tuples) == 0:
        return {"dice": np.nan, "accuracy": np.nan, "precision": np.nan, "recall": np.nan}
    arr = np.array(metric_tuples)
    return {
        "dice": float(np.mean(arr[:, 0])),
        "accuracy": float(np.mean(arr[:, 1])),
        "precision": float(np.mean(arr[:, 2])),
        "recall": float(np.mean(arr[:, 3])),
    }



# 5. Experimental Evaluation

### Without post-processing

For each experiment, mean Dice, accuracy, precision, and recall are computed on **oriented predictions** with no morphological operator (`Post_processing = No`).


### With post-processing

The **same oriented predictions** are passed through `apply_postprocessing()` (operator set by `POSTPROCESS_METHOD`) before metrics are computed (`Post_processing = Yes`). Compare operators by re-running the notebook with a different `POSTPROCESS_METHOD` and comparing exported CSV files.


# 7. Comparative Analysis

Build a single table comparing all experiments with and without post-processing.


## Evaluation configuration

### Configurable post-processing strategy

A **single** morphological operator is selected per notebook run via `POSTPROCESS_METHOD`. This preserves the lecturer comparison model (**No** vs **Yes**) while allowing systematic experiments across erosion, dilation, opening, and closing without expanding the results table.


In [ ]:
# ==========================================================
# EVALUATION CONFIGURATION
# ==========================================================

# Correct prediction orientation (as in Pos_Metrics.ipynb)
APPLY_ORIENTATION_CORRECTION = True

# Experiments to evaluate: (table label, results folder under 02_dataset/)
EXPERIMENTS_TO_EVALUATE = [
    ("Original", "results_original"),
    ("PP1", "results_pp_1"),
    ("PP2", "results_pp_2"),
    ("PP3", "results_pp_3"),
    ("PP4", "results_pp_4"),
    ("PP5", "results_pp_5"),
]

# Morphological method for the Yes branch (re-run notebook to compare methods)
POSTPROCESS_METHOD = "opening"

MORPH_KERNEL_SIZE = 3


## Path and pairing validation


In [ ]:
print("=" * 60)
print("PATH AND DATASET VALIDATION")
print("=" * 60)

warnings = validate_project_paths()
if warnings:
    for aviso in warnings:
        print(f"[AVISO] {aviso}")
else:
    print("Main folders found.")

print("\nPrediction ↔ label pairing (original identifier):")
pairing_report = validate_all_pairings()
for report_key, pairing_errors in pairing_report.items():
    if pairing_errors:
        print(f"  {report_key}: {len(pairing_errors)} error(s) — e.g. {pairing_errors[0]}")
    else:
        print(f"  {report_key}: OK")



In [ ]:
def evaluate_experiment(experiment_name: str, results_folder_name: str) -> list:
    """
    Evaluates all predictions in a results folder.
    Returns two metric rows: without and with post-processing.
    """
    results_folder = DATA_DIR / results_folder_name
    predictions = list_predictions(results_folder)

    if len(predictions) == 0:
        return [
            {
                "Experiment": experiment_name,
                "Post_processing": "No",
                "N": 0,
                "Dice": np.nan,
                "Accuracy": np.nan,
                "Precision": np.nan,
                "Recall": np.nan,
                "Pasta": results_folder_name,
            },
            {
                "Experiment": experiment_name,
                "Post_processing": "Yes",
                "N": 0,
                "Dice": np.nan,
                "Accuracy": np.nan,
                "Precision": np.nan,
                "Recall": np.nan,
                "Pasta": results_folder_name,
            },
        ]

    metrics_without_postprocess = []
    metrics_with_postprocess = []

    for prediction_path in predictions:
        label_path = resolve_label_path(prediction_path.name)

        predicted = load_mask_png(prediction_path)
        gt = load_mask_png(label_path)

        if APPLY_ORIENTATION_CORRECTION:
            predicted = correct_prediction_orientation(predicted)

        dice, ac, pr, re = calculate_metrics(predicted, gt)
        metrics_without_postprocess.append((dice, ac, pr, re))

        predicted_postprocessed = apply_postprocessing(predicted)
        dice, ac, pr, re = calculate_metrics(predicted_postprocessed, gt)
        metrics_with_postprocess.append((dice, ac, pr, re))

    mean_without_postprocess = compute_mean_metrics(metrics_without_postprocess)
    mean_with_postprocess = compute_mean_metrics(metrics_with_postprocess)

    return [
        {
            "Experiment": experiment_name,
            "Post_processing": "No",
            "N": len(predictions),
            "Dice": mean_without_postprocess["dice"],
            "Accuracy": mean_without_postprocess["accuracy"],
            "Precision": mean_without_postprocess["precision"],
            "Recall": mean_without_postprocess["recall"],
            "Pasta": results_folder_name,
        },
        {
            "Experiment": experiment_name,
            "Post_processing": "Yes",
            "N": len(predictions),
            "Dice": mean_with_postprocess["dice"],
            "Accuracy": mean_with_postprocess["accuracy"],
            "Precision": mean_with_postprocess["precision"],
            "Recall": mean_with_postprocess["recall"],
            "Pasta": results_folder_name,
        },
    ]


result_rows = []
for experiment_key, results_folder_key in EXPERIMENTS_TO_EVALUATE:
    result_rows.extend(evaluate_experiment(experiment_key, results_folder_key))

comparison_table = pd.DataFrame(result_rows)
column_order = [
    "Experiment",
    "Post_processing",
    "N",
    "Dice",
    "Accuracy",
    "Precision",
    "Recall",
    "Pasta",
]
comparison_table = comparison_table[column_order]



# 8. CSV Export

Persist the comparison table for the written report.


In [ ]:
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

display(comparison_table)

csv_filename = (
    f"tabela_avaliacao_experiencias_{POSTPROCESS_METHOD.lower()}.csv"
    if "POSTPROCESS_METHOD" in globals() and POSTPROCESS_METHOD
    else "tabela_avaliacao_experiencias_none.csv"
)
output_csv_path = PROJECT_ROOT / "04_pipeline_results" / csv_filename
output_csv_path.parent.mkdir(parents=True, exist_ok=True)
comparison_table.to_csv(output_csv_path, index=False)
print(f"\nTable saved to: {output_csv_path}")



# 7. Comparative Analysis (continued)

## Summary plot (Dice)


In [ ]:
if len(comparison_table) > 0 and comparison_table["Dice"].notna().any():
    fig, ax = plt.subplots(figsize=(10, 5))
    for postprocess_flag, group_df in comparison_table.groupby("Post_processing"):
        ax.plot(
            group_df["Experiment"],
            group_df["Dice"],
            marker="o",
            label=f"Post-processing: {postprocess_flag}",
        )
    ax.set_ylabel("Mean Dice")
    ax.set_xlabel("Experiment")
    ax.set_title(f"Mean Dice (POSTPROCESS_METHOD={POSTPROCESS_METHOD})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()



# 9. Conclusions

The comparison table summarises segmentation quality across preprocessing variants (No vs Yes for the configured morphological method).
Re-run with a different `POSTPROCESS_METHOD` to compare erosion, dilation, opening, and closing via separate CSV exports.
Use the CSV in `05_report/` and cross-check against `02_segmentation.ipynb` predictions.

**Generated artefact:** `04_pipeline_results/tabela_avaliacao_experiencias_{method}.csv` (suffix matches `POSTPROCESS_METHOD`)
